# Kaggle Launcher — E2VID Reconstruction + YOLO Training + Detection Cache
**Last updated: 2026-07-12  v60**

Three-session workflow — run sessions in order, each builds on the previous:

| Session | `SKIP_TRAINING` | `SKIP_DETECTION_CACHE` | `PREV_RECON_INPUT` | `PREV_WEIGHTS_INPUT` |
|---------|-----------------|------------------------|--------------------|----------------------|
| 1 · Recon | `True` | `True` | `None` | `None` |
| 2 · Train | `True`* | `True` | session 1 output | `None` |
| 3 · Cache | `True` | `False` | session 1 output | session 2 output |

*Set `SKIP_TRAINING=False` in session 2.

**Before running:**
1. Attach datasets: `gennepy/fred-events-ami`, `gennepy/fred-events-ami-2`, `gennepy/fred-scripts-ami`
2. Set Runtime → Accelerator → **GPU T4 x2**
3. Configure the cell below for the session you are running


## 1 · Configuration — edit this cell

In [ ]:
from pathlib import Path
import datetime

# ── Kaggle dataset paths ───────────────────────────────────────────────────────
AMI_INPUT        = Path('/kaggle/input/datasets/gennepy/fred-events-ami')
AMI_INPUT_2      = Path('/kaggle/input/datasets/gennepy/fred-events-ami-2')
SCRIPTS_INPUT    = Path('/kaggle/input/datasets/gennepy/fred-scripts-ami')
AMI_WORK         = Path('/kaggle/working')
PREV_RECON_INPUT    = None   # session 1 output — frames zip
PREV_WEIGHTS_INPUT  = None   # session 2 output — yolo_e2vid.pt

# ── Run control ───────────────────────────────────────────────────────────────
RESUME               = False
SKIP_TRAINING        = False
SKIP_DETECTION_CACHE = True   # set False in session 3

# ── Sequences ─────────────────────────────────────────────────────────────────
SEQUENCES     = [
    # Training (40)
    'sequence_0', 'sequence_5', 'sequence_14', 'sequence_18', 'sequence_25', 'sequence_28', 'sequence_33', 'sequence_38', 'sequence_44', 'sequence_49',
    'sequence_54', 'sequence_58', 'sequence_67', 'sequence_71', 'sequence_75', 'sequence_79', 'sequence_85', 'sequence_89', 'sequence_95', 'sequence_99',
    'sequence_104', 'sequence_108', 'sequence_116', 'sequence_120', 'sequence_124', 'sequence_131', 'sequence_135', 'sequence_140', 'sequence_145', 'sequence_149',
    'sequence_157', 'sequence_161', 'sequence_168', 'sequence_172', 'sequence_176', 'sequence_182', 'sequence_186', 'sequence_192', 'sequence_199', 'sequence_204',
    # Validation (10)
    'sequence_2', 'sequence_16', 'sequence_31', 'sequence_47', 'sequence_62', 'sequence_81', 'sequence_101', 'sequence_125', 'sequence_150', 'sequence_211',
    # Test canonical (5) — reconstructed but excluded from training
    'sequence_8', 'sequence_9', 'sequence_12', 'sequence_20', 'sequence_21',
]
VAL_SEQUENCES  = ['sequence_2', 'sequence_16', 'sequence_31', 'sequence_47', 'sequence_62', 'sequence_81', 'sequence_101', 'sequence_125', 'sequence_150', 'sequence_211']
TEST_SEQUENCES = ['sequence_8', 'sequence_9', 'sequence_12', 'sequence_20', 'sequence_21']

# ── Per-sequence start_s overrides ────────────────────────────────────────────
START_S_OVERRIDES = {'sequence_47': 7.0}
DEFAULT_START_S = 5.0

# ── Training parameters ───────────────────────────────────────────────────────
MODEL  = 'yolov8n.pt'
EPOCHS = 100
BATCH  = 16

# ── Reconstruction parameters ─────────────────────────────────────────────────
EVENTS_PER_PIXEL = 0.1
SMOKE_EVENTS     = None

# ── Derived paths ─────────────────────────────────────────────────────────────
SCRIPTS_DIR = SCRIPTS_INPUT
EVENTS_ROOT = AMI_INPUT / 'processed'
RAW_ROOT    = AMI_INPUT / 'raw'
RECON_ROOT  = AMI_WORK  / 'data' / 'processed'
WEIGHTS_OUT = AMI_WORK  / 'yolo_e2vid.pt'
RUNS_DIR    = AMI_WORK  / 'yolo_runs'
LOG_FILE    = AMI_WORK  / 'logs' / f'run_{datetime.datetime.now().strftime("%Y%m%d_%H%M%S")}.log'
DATASET_DIR = AMI_WORK  / 'yolo_e2vid'
WORK_DIR    = Path('/tmp/ami_work')

WORK_DIR.mkdir(parents=True, exist_ok=True)
LOG_FILE.parent.mkdir(parents=True, exist_ok=True)

train_seqs = [s for s in SEQUENCES if s not in VAL_SEQUENCES and s not in TEST_SEQUENCES]
print('Model                :', MODEL)
print('Train seqs           :', train_seqs)
print('Val seqs             :', VAL_SEQUENCES)
print('Epochs               :', EPOCHS, '  Batch:', BATCH)
print('Resume               :', RESUME)
print('Skip training        :', SKIP_TRAINING)
print('Skip detection cache :', SKIP_DETECTION_CACHE)
print('Prev recon input     :', PREV_RECON_INPUT)
print('Prev weights input   :', PREV_WEIGHTS_INPUT)
print('start_s              :', DEFAULT_START_S, ' overrides:', START_S_OVERRIDES)
print('Smoke events         :', SMOKE_EVENTS or 'full run')


## 2 · Install dependencies

In [ ]:
!pip install -q h5py ultralytics==8.4.54 imageio scikit-image pandas matplotlib
import torch
print(f'PyTorch {torch.__version__} — CUDA: {torch.cuda.is_available()}')

In [ ]:
import torch
if not torch.cuda.is_available():
    raise SystemExit(
        'No GPU detected. Go to Settings → Accelerator → GPU T4 x2, then restart.'
    )
print(f'GPU: {torch.cuda.get_device_name(0)}  '
      f'({torch.cuda.get_device_properties(0).total_memory // 1024**2} MB)')

## 3 · Helpers

In [ ]:
import os, subprocess, sys, datetime, shutil

# Copy scripts from input dataset to local working dir
LOCAL_SCRIPTS = Path('/kaggle/working/scripts')
LOCAL_SCRIPTS.mkdir(exist_ok=True)
for script in ['reconstruct.py', 'train_yolo.py']:
    shutil.copy(SCRIPTS_DIR / script, LOCAL_SCRIPTS / script)
print(f'Scripts copied to {LOCAL_SCRIPTS}')

def run_streaming(cmd):
    """Run a command, stream output to notebook and append to log file."""
    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'
    process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )
    with open(LOG_FILE, 'a') as lf:
        for line in process.stdout:
            print(line, end='', flush=True)
            lf.write(line)
            lf.flush()
    process.wait()
    return process.returncode

def log(msg):
    ts = datetime.datetime.now().strftime('%H:%M:%S')
    line = f'[{ts}] {msg}'
    print(line)
    with open(LOG_FILE, 'a') as f:
        f.write(line + '\n')

log('Helpers ready.')

## 4 · Cleanup — remove stale output before each run

In [ ]:
import shutil

if RESUME:
    print('RESUME=True — skipping cleanup, keeping existing outputs.')
else:
    for seq in SEQUENCES:
        recon_dir = RECON_ROOT / seq / 'reconstruction_e2vid'
        if recon_dir.exists():
            shutil.rmtree(recon_dir)
            print(f'Cleaned: {recon_dir}')
        else:
            print(f'Nothing to clean: {recon_dir}')

    shutil.rmtree(DATASET_DIR, ignore_errors=True)
    print(f'Cleaned: {DATASET_DIR}')

    shutil.rmtree(WORK_DIR / 'rpg_e2vid', ignore_errors=True)
    print(f'Cleaned: {WORK_DIR / "rpg_e2vid"}')

In [ ]:
# ── Restore previous reconstructions so they are not re-run ─────────────────
import shutil

if PREV_RECON_INPUT is not None and PREV_RECON_INPUT.exists():
    zip_candidates = sorted(PREV_RECON_INPUT.glob('*.zip'))
    if zip_candidates:
        import zipfile as _zf
        zip_file = zip_candidates[0]
        log(f'Extracting {zip_file.name} → {AMI_WORK} ...')
        with _zf.ZipFile(zip_file) as zf:
            zf.extractall(AMI_WORK)
        for seq in SEQUENCES:
            dst_dir = RECON_ROOT / seq / 'reconstruction_e2vid'
            if dst_dir.exists():
                n = len(list(dst_dir.glob('frame_*')))
                log(f'  {seq}: {n} frames extracted')
            else:
                log(f'  {seq}: not found in zip — will reconstruct')
    else:
        log('Restoring previous reconstructions from ' + str(PREV_RECON_INPUT))
        log('  Contents: ' + str([p.name for p in sorted(PREV_RECON_INPUT.iterdir())[:10]]))
        for seq in SEQUENCES:
            candidates = [
                PREV_RECON_INPUT / 'processed' / seq / 'reconstruction_e2vid',
                PREV_RECON_INPUT / 'data' / 'processed' / seq / 'reconstruction_e2vid',
                PREV_RECON_INPUT / seq,
            ]
            src_dir = next((p for p in candidates if p.exists()), None)
            dst_dir = RECON_ROOT / seq / 'reconstruction_e2vid'
            if src_dir is not None and not dst_dir.exists():
                dst_dir.parent.mkdir(parents=True, exist_ok=True)
                shutil.copytree(src_dir, dst_dir)
                n = len(list(dst_dir.glob('frame_*')))
                log(f'  {seq}: restored {n} frames')
            elif dst_dir.exists():
                log(f'  {seq}: already in working dir, skipping copy')
            else:
                log(f'  {seq}: not found in prev recon input — will reconstruct')
else:
    log('No PREV_RECON_INPUT — all sequences will be reconstructed from scratch')

## 5 · Reconstruct e2vid frames

In [ ]:
log('=== Reconstruction started ===')

for seq in SEQUENCES:
    zip_path = EVENTS_ROOT / seq / 'events.zip'
    out_dir  = RECON_ROOT  / seq / 'reconstruction_e2vid'

    if out_dir.exists() and (any(out_dir.glob('frame_*.png')) or any(out_dir.glob('frame_*.jpg'))):
        log(f'{seq}: frames already exist — skipping reconstruction')
        continue

    start_s = START_S_OVERRIDES.get(seq, DEFAULT_START_S)
    log(f'=== Reconstructing {seq} (start_s={start_s}, events_per_pixel={EVENTS_PER_PIXEL}) ===')
    cmd = [
        sys.executable, str(LOCAL_SCRIPTS / 'reconstruct.py'),
        '--zip_path',         str(zip_path),
        '--out_dir',          str(out_dir),
        '--work_dir',         str(WORK_DIR),
        '--events_per_pixel', str(EVENTS_PER_PIXEL),
        '--start_s',          str(start_s),
        '--compress_jpeg',
    ]
    if SMOKE_EVENTS:
        cmd += ['--max_events', str(SMOKE_EVENTS)]

    rc = run_streaming(cmd)
    if rc != 0:
        log(f'ERROR: reconstruct.py failed for {seq} (exit code {rc})')
        raise RuntimeError(f'reconstruct.py failed for {seq} (exit code {rc})')

log('=== Reconstruction done ===')

## 6 · Train YOLO

In [ ]:
if SKIP_TRAINING:
    log('SKIP_TRAINING=True — skipping training')
else:
    log('=== Training started ===')

    resume_yaml = DATASET_DIR / 'dataset.yaml'
    resume_yaml.parent.mkdir(parents=True, exist_ok=True)
    resume_yaml.write_text(f"""path: {DATASET_DIR}
train: train.txt
val:   val.txt

nc: 1
names:
  0: drone
""")
    log(f'dataset.yaml pre-written → {DATASET_DIR}')

    cmd = [
        sys.executable, str(LOCAL_SCRIPTS / 'train_yolo.py'),
        '--sequences',     *SEQUENCES,
        '--val_sequences', *VAL_SEQUENCES,
        '--raw_root',      str(RAW_ROOT),
        '--recon_root',    str(RECON_ROOT),
        '--out_dir',       str(DATASET_DIR),
        '--runs_dir',      str(RUNS_DIR),
        '--weights',       str(WEIGHTS_OUT),
        '--model',         MODEL,
        '--epochs',        str(EPOCHS),
        '--batch',         str(BATCH),
    ]

    if RESUME:
        cmd += ['--resume']

    rc = run_streaming(cmd)
    if rc != 0:
        log(f'ERROR: train_yolo.py failed (exit code {rc})')
        raise RuntimeError('train_yolo.py failed')

    log(f'=== Training done — weights at {WEIGHTS_OUT} ===')

## 7 · Cache detections

Runs the trained YOLO model over all reconstructed frames and saves
`detections_e2vid.json` per sequence. Weights come from this session's
training output or from `PREV_WEIGHTS_INPUT` (session 3 standalone run).

On a T4 GPU with batch=32, this takes ~10–20 min for 230 sequences.


In [ ]:
import json
from ultralytics import YOLO

if SKIP_DETECTION_CACHE:
    log('SKIP_DETECTION_CACHE=True — skipping')
else:
    # Resolve weights: prefer this session's output, fall back to PREV_WEIGHTS_INPUT
    weights_path = WEIGHTS_OUT
    if not weights_path.exists() and PREV_WEIGHTS_INPUT is not None:
        for candidate in [
            PREV_WEIGHTS_INPUT / 'yolo_e2vid.pt',
            PREV_WEIGHTS_INPUT / 'yolo_runs' / 'e2vid' / 'weights' / 'best.pt',
        ]:
            if candidate.exists():
                weights_path = candidate
                log(f'  weights loaded from PREV_WEIGHTS_INPUT: {candidate}')
                break

    if not weights_path.exists():
        raise FileNotFoundError(
            f'No weights found. Set PREV_WEIGHTS_INPUT or run training first.\n'
            f'Looked at: {WEIGHTS_OUT} and PREV_WEIGHTS_INPUT={PREV_WEIGHTS_INPUT}'
        )

    log(f'=== Detection caching started — weights: {weights_path} ===')
    model = YOLO(str(weights_path))
    INFER_BATCH = 32
    total_seqs = 0
    total_dets = 0

    for seq in SEQUENCES:
        recon_dir = RECON_ROOT / seq / 'reconstruction_e2vid'
        if not recon_dir.exists():
            log(f'  {seq}: no frames — skipping')
            continue

        frames = sorted(
            list(recon_dir.glob('frame_*.jpg')) + list(recon_dir.glob('frame_*.png')),
            key=lambda f: int(f.stem.split('_')[-1])
        )
        if not frames:
            log(f'  {seq}: no frames — skipping')
            continue

        detections = []
        for i in range(0, len(frames), INFER_BATCH):
            batch = frames[i:i + INFER_BATCH]
            results = model([str(f) for f in batch], verbose=False, conf=0.1)
            for frame_path, result in zip(batch, results):
                frame_num = int(frame_path.stem.split('_')[-1])
                for box in result.boxes:
                    x1, y1, x2, y2 = box.xyxy[0].tolist()
                    detections.append({
                        'frame':      frame_num,
                        'bbox':       [x1, y1, x2 - x1, y2 - y1],
                        'confidence': float(box.conf[0]),
                        'class':      'drone',
                    })

        cache = {'sequence_id': seq, 'model': 'e2vid', 'cached': True, 'detections': detections}
        cache_path = RECON_ROOT / seq / 'detections_e2vid.json'
        cache_path.write_text(json.dumps(cache))
        total_dets += len(detections)
        total_seqs += 1
        log(f'  {seq}: {len(frames)} frames → {len(detections)} detections')

    log(f'=== Detection caching done — {total_seqs} sequences, {total_dets} total detections ===')



## 8 · Zip reconstruction frames + detection cache for download

Bundles frames (jpg) + timestamps.txt + detections_e2vid.json into a single zip.
Safe to re-run: overwrites the zip if it already exists.


In [ ]:
import zipfile, json
from pathlib import Path

zip_path = AMI_WORK / 'frames_and_detections.zip'
log('=== Zipping frames + detections ===')
frame_count = 0
det_count = 0

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_STORED) as zf:
    for seq in SEQUENCES:
        recon_dir = RECON_ROOT / seq / 'reconstruction_e2vid'
        if not recon_dir.exists():
            log(f'  {seq}: no frames found, skipping')
            continue

        # timestamps.txt
        ts_file = recon_dir / 'timestamps.txt'
        if ts_file.exists():
            zf.write(ts_file, ts_file.relative_to(AMI_WORK))

        # detections JSON (if caching was run)
        det_file = RECON_ROOT / seq / 'detections_e2vid.json'
        if det_file.exists():
            zf.write(det_file, det_file.relative_to(AMI_WORK))
            det_count += 1

        # frames (stream-and-delete to keep disk flat)
        frames = sorted(recon_dir.glob('frame_*.jpg'))
        for frame in frames:
            zf.write(frame, frame.relative_to(AMI_WORK))
            frame.unlink()
            frame_count += 1
        log(f'  {seq}: {len(frames)} frames' + (', detections cached' if det_file.exists() else ''))

size_mb = zip_path.stat().st_size / 1e6
log(f'frames_and_detections.zip: {frame_count} frames, {det_count} detection files, {size_mb:.0f} MB')


In [ ]:
# Free disk space so outputs fit in the Kaggle snapshot.
import shutil, os

freed = 0

def _rmdir(p):
    global freed
    p = Path(p)
    if p.exists():
        size = sum(f.stat().st_size for f in p.rglob('*') if f.is_file())
        shutil.rmtree(p)
        freed += size
        log(f'  deleted {p}  ({size / 1e9:.2f} GB)')

def _rm(p):
    global freed
    p = Path(p)
    if p.is_file():
        freed += p.stat().st_size
        p.unlink()

# ── Save YOLO outputs to kpis/ before cleanup ────────────────────────────────
e2vid_run = RUNS_DIR / 'e2vid'
kpis_dir  = AMI_WORK / 'kpis'
kpis_dir.mkdir(parents=True, exist_ok=True)

KEEP = {
    'results.csv', 'results.png',
    'confusion_matrix.png', 'confusion_matrix_normalized.png',
    'BoxF1_curve.png', 'BoxP_curve.png', 'BoxR_curve.png', 'BoxPR_curve.png',
    'labels.jpg', 'labels_correlogram.jpg',
    'train_batch0.jpg', 'train_batch1.jpg', 'train_batch2.jpg',
    'val_batch0_labels.jpg', 'val_batch0_pred.jpg',
    'val_batch1_labels.jpg', 'val_batch1_pred.jpg',
    'val_batch2_labels.jpg', 'val_batch2_pred.jpg',
}

if e2vid_run.exists():
    for p in e2vid_run.iterdir():
        if p.is_file() and p.name in KEEP:
            shutil.copy(p, kpis_dir / p.name)
            log(f'  saved {p.name} → kpis/')

# ── Delete large directories ──────────────────────────────────────────────────
_rmdir(RECON_ROOT.parent)

if e2vid_run.exists():
    for p in list(e2vid_run.iterdir()):
        if p.name != 'weights':
            _rmdir(p) if p.is_dir() else _rm(p)

_rmdir(WORK_DIR)

log(f'Freed {freed / 1e9:.2f} GB total')
os.system('df -h /kaggle/working')

## 9 · Download results

After the notebook finishes, `/kaggle/working/` is saved as the session output.

**Session 1 — frames zip (~15 GB for 230 seqs):**
```bash
bash scripts/sync_from_kaggle.sh --frames-zip
```

**Session 2 — weights + KPIs (~50 MB):**
```bash
bash scripts/sync_from_kaggle.sh
```

**Session 3 — detections zip (tiny, ~30 MB for 230 seqs):**
```bash
bash scripts/sync_from_kaggle.sh --detections-zip
```
Extracts `detections_e2vid.json` per sequence into `data/processed/`.

**Rebuild and push e2vid service after session 2:**
```bash
cp data/yolo_runs/e2vid/weights/best.pt services/e2vid/weights/yolo_e2vid.pt
docker build -t ghcr.io/gennepy/ami-e2vid:latest services/e2vid/
gh auth token | docker login ghcr.io -u gennepy --password-stdin
docker push ghcr.io/gennepy/ami-e2vid:latest
```
